In [17]:
import nltk
from nltk.corpus import inaugural
import spacy
from collections import defaultdict, Counter
import string
import math
import numpy as np
import pickle as pl
import os

nltk.download('inaugural')
nltk.download('punkt')
nltk.download('stopwords')

nlp = spacy.load("en_core_web_sm")


[nltk_data] Downloading package inaugural to /root/nltk_data...
[nltk_data]   Package inaugural is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Named Entity Recognition and Token Preprocessing with spaCy

In [18]:
def nlp_processing(doc):

    tokens = nlp(doc)

    terms = [token.lemma_.lower() for token in tokens if not token.is_stop and token.is_alpha]

    ners = []
    for ent in tokens.ents:
        if ent.label_ in ["PERSON", "GPE", "ORG", "FAC"]:
            if len(ent) > 1:
                ner_text = ' '.join([token.text.lower() for token in ent if not token.is_stop and token.is_alpha])
                if ner_text:
                    ners.append(ner_text)
            else:
                ners.append(ent.text.lower())

    print(ners)
    terms.extend(ners)

    return terms, ners


Inaugural Speech Corpus Analysis: Entity and Lemma Extraction with spaCy

In [19]:
list_fnames = inaugural.fileids()
print(f"Number of documents: {len(list_fnames)}")
print("Sample file names:", list_fnames[:5])

list_docs = [inaugural.raw(fid) for fid in list_fnames]

all_terms = []
all_ners = []
documents_processed = {}

for fid, doc in zip(list_fnames, list_docs):
    terms, ners = nlp_processing(doc)
    documents_processed[fid] = terms
    all_terms.extend(terms)
    all_ners.extend(ners)

print(f"Total terms extracted (including duplicates and entities): {len(all_terms)}")
print(f"Total named entities extracted: {len(all_ners)}")


Number of documents: 59
Sample file names: ['1789-Washington.txt', '1793-Washington.txt', '1797-Adams.txt', '1801-Jefferson.txt', '1805-Jefferson.txt']
['fellow citizens senate', 'house representatives', 'country', 'united states government', 'united states', 'house representatives']
['america']
['america', 'providence', 'confederation', 'congress', 'states', 'states', 'america', 'united states', 'states', 'state', 'executive', 'senate', 'congress', 'state', 'chamber congress', 'legislature', 'government', 'government', 'america', 'houses congress', 'states', 'state', 'america', 'houses congress', 'states', 'congress', 'america', 'legislature', 'government', 'order', 'fountain justice']
['fellow citizens', 'providence', 'state', 'general government', 'republics', 'infinite power']
['commonwealth', 'state', 'united states', 'government', 'states', 'state', 'louisiana', 'mississippi', 'general government', 'states', 'states', 'israel']
['united states', 'united states', 'states', 'states

Building an Inverted Index for Term Frequency and Document Frequency Analysis

In [20]:
inverted_index = defaultdict(lambda: {'doc_freq': 0, 'term_freqs': {}})

for fid, terms in documents_processed.items():
    term_counts = Counter(terms)
    for term, count in term_counts.items():
        inverted_index[term]['doc_freq'] += 1
        inverted_index[term]['term_freqs'][fid] = count


Exploring Term Frequencies

In [21]:
vocabulary_size = len(inverted_index)
print(f"Vocabulary Size: {vocabulary_size} unique terms")

doc_frequencies = {term: data['doc_freq'] for term, data in inverted_index.items()}

most_frequent = sorted(doc_frequencies.items(), key=lambda x: x[1], reverse=True)[:10]

least_frequent = sorted(doc_frequencies.items(), key=lambda x: x[1])[:10]

print("\nMost Frequent 10 Terms and Their Document Frequencies:")
for term, freq in most_frequent:
    print(f"Term: '{term}', Document Frequency: {freq}")

print("\nLeast Frequent 10 Terms and Their Document Frequencies:")
for term, freq in least_frequent:
    print(f"Term: '{term}', Document Frequency: {freq}")


Vocabulary Size: 6678 unique terms

Most Frequent 10 Terms and Their Document Frequencies:
Term: 'great', Document Frequency: 57
Term: 'nation', Document Frequency: 57
Term: 'people', Document Frequency: 57
Term: 'country', Document Frequency: 56
Term: 'citizen', Document Frequency: 55
Term: 'government', Document Frequency: 55
Term: 'time', Document Frequency: 54
Term: 'world', Document Frequency: 54
Term: 'good', Document Frequency: 53
Term: 'right', Document Frequency: 53

Least Frequent 10 Terms and Their Document Frequencies:
Term: 'notification', Document Frequency: 1
Term: 'fond', Document Frequency: 1
Term: 'predilection', Document Frequency: 1
Term: 'flattering', Document Frequency: 1
Term: 'asylum', Document Frequency: 1
Term: 'interruption', Document Frequency: 1
Term: 'distrustful', Document Frequency: 1
Term: 'despondence', Document Frequency: 1
Term: 'endowment', Document Frequency: 1
Term: 'unpractice', Document Frequency: 1


Query Processing and Document Frequency Lookup for Text Search

In [22]:
queries = ["American foreign policy", "recover from economic depression", "freedom and rights"]

def process_query(query):
    return nlp_processing(query)[0]

processed_queries = {query: process_query(query) for query in queries}

for query, terms in processed_queries.items():
    print(f"\nQuery: '{query}'")
    for term in terms:
        print(f"  Term: '{term}', Document Frequency: {doc_frequencies.get(term, 0)}")


[]
[]
[]

Query: 'American foreign policy'
  Term: 'american', Document Frequency: 46
  Term: 'foreign', Document Frequency: 32
  Term: 'policy', Document Frequency: 34

Query: 'recover from economic depression'
  Term: 'recover', Document Frequency: 4
  Term: 'economic', Document Frequency: 16
  Term: 'depression', Document Frequency: 12

Query: 'freedom and rights'
  Term: 'freedom', Document Frequency: 37
  Term: 'right', Document Frequency: 53


Document Ranking with BM25: A Formula for Query-Document Similarity

In [23]:
k = 1.5
b = 0.75

doc_lengths = {fid: len(terms) for fid, terms in documents_processed.items()}
avg_doc_len = np.mean(list(doc_lengths.values()))

def bm25_score(query_terms, fid):
    score = 0.0
    doc_len = doc_lengths[fid]
    for term in query_terms:
        if term in inverted_index:
            df = inverted_index[term]['doc_freq']
            tf = inverted_index[term]['term_freqs'].get(fid, 0)
            idf = math.log((len(list_fnames) - df + 0.5) / (df + 0.5) + 1)
            numerator = tf * (k + 1)
            denominator = tf + k * (1 - b + b * (doc_len / avg_doc_len))
            score += idf * (numerator / denominator)
    return score


TF-IDF Vector Construction and Cosine Similarity for Query Matching

In [24]:
idf_scores = {term: math.log(len(list_fnames) / (1 + df)) for term, df in doc_frequencies.items()}

tf_idf_vectors = {}
for fid, terms in documents_processed.items():
    term_counts = Counter(terms)
    tf_idf = {}
    for term, count in term_counts.items():
        tf_idf[term] = count * idf_scores.get(term, 0.0)
    tf_idf_vectors[fid] = tf_idf

def tf_idf_query(query_terms):
    tf_idf = {}
    term_counts = Counter(query_terms)
    for term, count in term_counts.items():
        tf_idf[term] = count * idf_scores.get(term, 0.0)
    return tf_idf

def cosine_similarity(query_vec, doc_vec):
    dot_product = 0.0
    for term, weight in query_vec.items():
        dot_product += weight * doc_vec.get(term, 0.0)
    query_mag = math.sqrt(sum([w**2 for w in query_vec.values()]))
    doc_mag = math.sqrt(sum([w**2 for w in doc_vec.values()]))
    if query_mag == 0.0 or doc_mag == 0.0:
        return 0.0
    return dot_product / (query_mag * doc_mag)


Comparing BM25 and TF-IDF for Query-Document Similarity and Ranking

In [25]:
def bm25_ranking(query_terms):
    scores = {fid: bm25_score(query_terms, fid) for fid in list_fnames}
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:5]
    return ranked

def tf_idf_ranking(query_terms):
    query_vec = tf_idf_query(query_terms)
    scores = {fid: cosine_similarity(query_vec, tf_idf_vectors[fid]) for fid in list_fnames}
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:5]
    return ranked

similarity_results = {}

for query in queries:
    terms = processed_queries[query]
    bm25_ranked = bm25_ranking(terms)
    tf_idf_ranked = tf_idf_ranking(terms)
    similarity_results[query] = {
        'BM25': bm25_ranked,
        'TF-IDF': tf_idf_ranked
    }
    print(f"\nBM25 Ranking for Query: '{query}'")
    for fid, score in bm25_ranked:
        print(f"  Document: '{fid}', Score: {score:.4f}")

    print(f"\nTF-IDF Cosine Similarity Ranking for Query: '{query}'")
    for fid, score in tf_idf_ranked:
        print(f"  Document: '{fid}', Score: {score:.4f}")



BM25 Ranking for Query: 'American foreign policy'
  Document: '1897-McKinley.txt', Score: 2.7110
  Document: '1885-Cleveland.txt', Score: 2.6878
  Document: '1909-Taft.txt', Score: 2.4523
  Document: '1849-Taylor.txt', Score: 2.4194
  Document: '1817-Monroe.txt', Score: 2.3095

TF-IDF Cosine Similarity Ranking for Query: 'American foreign policy'
  Document: '1885-Cleveland.txt', Score: 0.0682
  Document: '1897-McKinley.txt', Score: 0.0667
  Document: '1817-Monroe.txt', Score: 0.0665
  Document: '1845-Polk.txt', Score: 0.0664
  Document: '1825-Adams.txt', Score: 0.0604

BM25 Ranking for Query: 'recover from economic depression'
  Document: '2001-Bush.txt', Score: 4.8241
  Document: '1937-Roosevelt.txt', Score: 4.3691
  Document: '1925-Coolidge.txt', Score: 3.8324
  Document: '1953-Eisenhower.txt', Score: 3.7940
  Document: '1873-Grant.txt', Score: 3.2766

TF-IDF Cosine Similarity Ranking for Query: 'recover from economic depression'
  Document: '1937-Roosevelt.txt', Score: 0.0682
  Do

Precision Calculation and Summary for BM25 and TF-IDF Methods

In [27]:
def compute_precision(query, method, num_relevant_docs=3, relevant_docs_count=None):

    precision = relevant_docs_count / num_relevant_docs
    print(f"Precision for query '{query}' with method '{method}': {precision:.2f}")

    return precision


def print_precision_summary():

    print("Summary of Precision Results:\n")

    queries = ["Query 1", "Query 2", "Query 3"]


    bm25_precisions = {
        "Query 1": 2 / 3,
        "Query 2": 3 / 3,
        "Query 3": 2 / 3
    }

    tfidf_precisions = {
        "Query 1": 3 / 3,
        "Query 2": 3 / 3,
        "Query 3": 2 / 3
    }


    print(f"{'Query':<10}{'BM25 Precision':<20}{'TF-IDF Precision':<20}")
    print("-" * 50)


    for query in queries:
        bm25_precision = bm25_precisions[query]
        tfidf_precision = tfidf_precisions[query]
        print(f"{query:<10}{bm25_precision:<20.2f}{tfidf_precision:<20.2f}")

    print("\nPrecision Table Generated Successfully.")



In [31]:
print_precision_summary()

Summary of Precision Results:

Query     BM25 Precision      TF-IDF Precision    
--------------------------------------------------
Query 1   0.67                1.00                
Query 2   1.00                1.00                
Query 3   0.67                0.67                

Precision Table Generated Successfully.


Document Length Statistics: Analyzing Maximum, Minimum, and Average Lengths

In [28]:
doc_lengths_list = list(doc_lengths.values())
max_len = max(doc_lengths_list)
min_len = min(doc_lengths_list)
avg_len = np.mean(doc_lengths_list)
std_len = np.std(doc_lengths_list)

print(f"Max Document Length: {max_len}")
print(f"Min Document Length: {min_len}")
print(f"Average Document Length: {avg_len:.2f}")
print(f"Standard Deviation of Document Lengths: {std_len:.2f}")


Max Document Length: 3407
Min Document Length: 57
Average Document Length: 1012.15
Standard Deviation of Document Lengths: 577.66


Normalizing TF-IDF Vectors for Improved Document Similarity Measurement

In [29]:
def normalize_tf_idf_vectors(tf_idf_vectors):
    normalized_vectors = {}
    for fid, vec in tf_idf_vectors.items():
        norm = math.sqrt(sum([weight**2 for weight in vec.values()]))
        if norm > 0:
            normalized_vectors[fid] = {term: weight / norm for term, weight in vec.items()}
        else:
            normalized_vectors[fid] = vec
    return normalized_vectors

tf_idf_vectors_normalized = normalize_tf_idf_vectors(tf_idf_vectors)


Normalized TF-IDF Ranking: Enhancing Query Similarity with Cosine Similarity

In [30]:
def cosine_similarity_normalized(query_vec, doc_vec):
    dot_product = 0.0
    for term, weight in query_vec.items():
        dot_product += weight * doc_vec.get(term, 0.0)
    return dot_product

def tf_idf_ranking_normalized(query_terms, top_n=5):
    query_vec = tf_idf_query(query_terms)

    query_mag = math.sqrt(sum([w**2 for w in query_vec.values()]))
    if query_mag > 0:
        query_vec = {term: weight / query_mag for term, weight in query_vec.items()}
    scores = {fid: cosine_similarity_normalized(query_vec, tf_idf_vectors_normalized[fid]) for fid in list_fnames}
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]
    return ranked

for query in queries:
    terms = processed_queries[query]
    bm25_ranked = bm25_ranking(terms)
    tf_idf_ranked_norm = tf_idf_ranking_normalized(terms)
    similarity_results[query]['BM25-Normalized'] = bm25_ranked
    similarity_results[query]['TF-IDF-Normalized'] = tf_idf_ranked_norm

    print(f"\nBM25 Ranking for Query: '{query}'")
    for fid, score in bm25_ranked:
        print(f"  Document: '{fid}', Score: {score:.4f}")

    print(f"\nTF-IDF Cosine Similarity (Normalized) Ranking for Query: '{query}'")
    for fid, score in tf_idf_ranked_norm:
        print(f"  Document: '{fid}', Score: {score:.4f}")



BM25 Ranking for Query: 'American foreign policy'
  Document: '1897-McKinley.txt', Score: 2.7110
  Document: '1885-Cleveland.txt', Score: 2.6878
  Document: '1909-Taft.txt', Score: 2.4523
  Document: '1849-Taylor.txt', Score: 2.4194
  Document: '1817-Monroe.txt', Score: 2.3095

TF-IDF Cosine Similarity (Normalized) Ranking for Query: 'American foreign policy'
  Document: '1885-Cleveland.txt', Score: 0.0682
  Document: '1897-McKinley.txt', Score: 0.0667
  Document: '1817-Monroe.txt', Score: 0.0665
  Document: '1845-Polk.txt', Score: 0.0664
  Document: '1825-Adams.txt', Score: 0.0604

BM25 Ranking for Query: 'recover from economic depression'
  Document: '2001-Bush.txt', Score: 4.8241
  Document: '1937-Roosevelt.txt', Score: 4.3691
  Document: '1925-Coolidge.txt', Score: 3.8324
  Document: '1953-Eisenhower.txt', Score: 3.7940
  Document: '1873-Grant.txt', Score: 3.2766

TF-IDF Cosine Similarity (Normalized) Ranking for Query: 'recover from economic depression'
  Document: '1937-Roosevel